In [9]:
import pandas as pd
import sqlite3

# Connect to the database you created in Phase 2
conn = sqlite3.connect('pharma_sfe.db')

# Load the tables we need to clean
calls = pd.read_sql("SELECT * FROM calls", conn)
reps = pd.read_sql("SELECT * FROM reps", conn)
prescribers = pd.read_sql("SELECT * FROM prescribers", conn)

In [13]:
# Check how many we have before
print(f"Rows before: {len(calls)}")

# Drop duplicates based on call_id or all columns
calls = calls.drop_duplicates()

print(f"Rows after: {len(calls)}") # Should be exactly 50,000

Rows before: 50000
Rows after: 50000


In [14]:
# Join calls with reps to get the territory_id for each call
calls_merged = calls.merge(reps[['rep_id', 'territory_id']], on='rep_id', how='left')

# Fill missing feedback_score with the median of that territory
calls_merged['feedback_score'] = calls_merged.groupby('territory_id')['feedback_score'].transform(
    lambda x: x.fillna(x.median())
)

# Final check: No more nulls!
print(calls_merged['feedback_score'].isnull().sum())

0


In [15]:
# Recalculate tiers based on actual volume percentiles
prescribers['prescriber_tier'] = pd.qcut(
    prescribers['monthly_rx_volume'], 
    3, 
    labels=['C', 'B', 'A']
)

In [16]:
# Connect to a NEW database for processed data
clean_conn = sqlite3.connect('processed_pharma_sfe.db')

# Save cleaned tables
calls_merged.to_sql('calls_cleaned', clean_conn, if_exists='replace', index=False)
prescribers.to_sql('prescribers_cleaned', clean_conn, if_exists='replace', index=False)

clean_conn.close()
print("Cleaning complete! Processed data saved.")

Cleaning complete! Processed data saved.
